In [1]:
import sys
import time
import math

# Fisrt, we add the location of the library to test to the PYTHON path
from liblibra_core import *
from libra_py import units as units
from libra_py import influence_spectrum as infsp
from libra_py import data_visualize
from libra_py import data_conv
from libra_py import data_stat
import libra_py.workflows.nbra.step4 as step4
import libra_py.workflows.nbra.lz as lz
import libra_py.workflows.nbra.decoherence_times as decoherence_times
import numpy as np
import matplotlib.pyplot as plt
import os
import multiprocessing as mp



<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for boost::python::detail::container_element<std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > >, unsigned long, boost::python::detail::final_vector_derived_policies<std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > >, false> > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for std::vector<std::vector<float, std::allocator<float> >, std::allocator<std::vector<float, std::allocator<float> > > > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWar

In [2]:

####################
# 1. Read / Get the Hvib files from step3/ci_zn-cspbi3
#  
# For the SD (SP) case
import os
import re
from liblibra_core import CMATRIX

print("\nGathering data from MD")

absolute_path = os.getcwd()
data_path = os.path.join(absolute_path, "3_step3", "ci_zn-cspbi3")

nfiles = 999
nstates = 11

# Regular expression for (real,imag)
pattern = re.compile(r"\(([-+0-9Ee\.]+),([-+0-9Ee\.]+)\)")

hvib_mixed_sd = []

for step in range(nfiles):

    filename = os.path.join(data_path, f"hvib_adi_{step}.txt")

    H = CMATRIX(nstates, nstates)

    with open(filename, "r") as f:
        for i, line in enumerate(f):
            vals = pattern.findall(line)

            for j, (re_part, im_part) in enumerate(vals):
                H.set(i, j, float(re_part), float(im_part))

    hvib_mixed_sd.append(H)

print("Number of Hvib matrices =", len(hvib_mixed_sd))

hvib_mixed_sd[-1].show_matrix()







Gathering data from MD
Number of Hvib matrices = 999
(0.0000000,0.0000000)  (0.0000000,-0.00044880739)  (0.0000000,-0.00069705378)  (0.0000000,-0.0010796118)  (0.0000000,-0.00030796219)  (0.0000000,0.00039085843)  (0.0000000,0.0010199902)  (0.0000000,-0.00056891974)  (0.0000000,0.00062270449)  (0.0000000,0.0012393884)  (0.0000000,-0.00011659686)  
(0.0000000,0.00044880739)  (0.061791739,0.0000000)  (0.0000000,-0.00011129230)  (0.0000000,-0.00046618636)  (0.0000000,-0.0010987561)  (0.0000000,-0.00043595946)  (0.0000000,0.00055801717)  (0.0000000,0.00023998565)  (0.0000000,0.00055154789)  (0.0000000,-2.1131279e-05)  (0.0000000,-9.6273296e-05)  
(0.0000000,0.00069705378)  (0.0000000,0.00011129230)  (0.066519606,0.0000000)  (0.0000000,2.7447950e-05)  (0.0000000,-0.0014005216)  (0.0000000,-0.00024546027)  (0.0000000,0.00027840949)  (0.0000000,0.00057173164)  (0.0000000,0.00027539388)  (0.0000000,-0.00099858521)  (0.0000000,0.00045079987)  
(0.0000000,0.0010796118)  (0.0000000,0.00046618636

In [3]:
def compute_state_energies_vs_time( hvib ):     
    """
    Computes the states energies vs time for a given hvib.
        hvib ( list of list of matrices ): The vibronic hamiltonian for all timesteps
    returns: a list of energies vs time for all states in hvib
    """
    nsteps   = len(hvib) 
    nstates  = hvib[0].num_of_rows
    energies = []
    for state in range( nstates ):
        energies.append( [] )
        for step in range( nsteps ):
            energies[ state ].append( hvib[ step ].get( state, state ).real - hvib[ step ].get( 0, 0 ).real )
    return np.array( energies )


In [4]:
def run_nbra_namd_wrapper( hvib, istate, outfile_name, params ):
    """
    A small wrapper function to run the nbra namd dynamics. 
        hvib ( list of list of matrices ): The vibronic hamiltonian for all timesteps
        istate ( int ): Index of the initial state. This index is from 0
        outfile ( string ): The name of the output file
        params ( dict ): A dictionary of important dynamical parameters
    returns: A list of energy values that is the electronic energy vs time. Returned energy is in eV
    """

    print("decoherence_method = ", params["decoherence_method"])
  
    params["outfile"]  = outfile_name
    params["nstates"]  = hvib[0].num_of_rows
    params["istate"]   = istate
    res_nbra_namd = step4.run( [ hvib ], params )
    nbra_namd     = data_conv.MATRIX2nparray( res_nbra_namd )
    # For SH energies
    energy_nbra_namd = ( nbra_namd[:,3*params["nstates"]+2] - nbra_namd[:,1] )*units.au2ev
    return energy_nbra_namd





In [5]:
def run_bllz_wrapper( hvib, istate, outfile_name, params ):
    """
    A small wrapper function to run the bllz dynamics. 
        hvib ( list of list of matrices ): The vibronic hamiltonian for all timesteps
        istate ( int ): Index of the initial state. This index is from 0
        outfile ( string ): The name of the output file
        params ( dict ): A dictionary of important dynamical parameters
    returns: A list of energy values that is the electronic energy vs time. Returned energy is in eV
    """

    params["outfile"]  = outfile_name
    params["nstates"]  = hvib[0].num_of_rows
    params["istate"]   = istate
    res_bllz, P = lz.run( [hvib], params )
    bllz_namd   = data_conv.MATRIX2nparray( res_bllz )
    # Recall that for bllz, we use the schrodiger not the surface hopping energy
    energy_bllz = ( bllz_namd[:,3*params["nstates"]+1] - bllz_namd[:,1] )*units.au2ev
    # for bllz surface hopping energy
    #energy_bllz = ( bllz_namd[:,3*params["nstates"]+2] - bllz_namd[:,1] )*units.au2ev
    return energy_bllz


In [6]:
def get_initial_states( energies, num_istates_to_generate, prefactor, average_excitation_energy, ref_state ):
    """
    This function gets initial states for a given electronic energies vs. time
    """
   
    nstates = len(energies)    
    # Obtain energy of first step for each electronic states
    initial_excitation_energies = []
    for state in range( nstates ):
        ref_energy = energies[ref_state][0]
        initial_excitation_energy = ( energies[state][0] - ref_energy ) * units.au2ev
        initial_excitation_energies.append( initial_excitation_energy  )
        print( "State", state, initial_excitation_energy )

    rnd = Random()
    excitation_energies = [] 
    # Compute energy window for a given E_avg
    for i in range( num_istates_to_generate ):
        ksi = prefactor * rnd.normal()
        excitation_energies.append( average_excitation_energy + ksi )
    print(excitation_energies)
   
    istates = []
    for i in range( num_istates_to_generate ):
        index = min( range(len(initial_excitation_energies)), key=lambda j: abs(initial_excitation_energies[j]-excitation_energies[i]))
        istates.append( index )

    return istates

In [7]:
#####################
# 2. Divide up into many nuclear trajectories. These are to be consdiered our independent nuclear trajectories
#####################

nuclear_traj_parser = [
    [0,0],
    [0,199],
    [0,399],
    [0,599],
    [0,799]
]

num_nuclear_trajs = len(nuclear_traj_parser)

# choose a window length that fits
nuclear_traj_len = 200

dt = 1.0 * units.fs2au

hvib_mixed_sd_trajs = []

# Number of electronic states
nstates_sp = hvib_mixed_sd[0].num_of_rows

for i in range(num_nuclear_trajs):
    hvib_mixed_sd_trajs.append([])

nsteps = len(hvib_mixed_sd)

for traj in range(num_nuclear_trajs):

    start_time = nuclear_traj_parser[traj][1]
    end_time = start_time + nuclear_traj_len

    if end_time > nsteps:
        print(f"Skipping trajectory {traj}: exceeds trajectory length.")
        continue

    hvib_mixed_sd_trajs[traj] = hvib_mixed_sd[start_time:end_time]


In [8]:
#####################
# 3. For each nuclear trajectory, we need to get the state energies vs. time, which is needed to get the istates
#istates_mb_all_trajs  = []
#energies_mb_all_trajs = []
istates_mixed_sd_all_trajs  = []
energies_mixed_sd_all_trajs = []

num_istates_to_generate = 500
for traj in range( num_nuclear_trajs ):
    
    #energies_mb_all_trajs.append( compute_state_energies_vs_time( hvib_mb_trajs[ traj ] ) )
    #istates_mb_all_trajs.append( get_initial_states( energies_mb_all_trajs[ traj ], num_istates_to_generate, 0.01, 0.4, 0 ) )

    energies_mixed_sd_all_trajs.append( compute_state_energies_vs_time( hvib_mixed_sd_trajs[ traj ] ) )
    istates_mixed_sd_all_trajs.append( get_initial_states( energies_mixed_sd_all_trajs[ traj ], num_istates_to_generate, 0.01, 0.4, 0 ) )

#print(istates_mb_all_trajs)
print(istates_mixed_sd_all_trajs)
#sys.exit(0

State 0 0.0
State 1 1.8079800104129997
State 2 1.8248400004349998
State 3 1.8733999891270001
State 4 1.8957099887059998
State 5 1.9545699939619998
State 6 1.9582500096019997
State 7 1.968879986752
State 8 2.022179995589
State 9 2.057480009559
State 10 2.0980500051099997
[0.39793051459109097, 0.40817708034223094, 0.4031631079071709, 0.41217023605844516, 0.3982540010492837, 0.4004547859808427, 0.3914818823253857, 0.38642730950392923, 0.41265222500793, 0.3918326685646561, 0.40343496756695124, 0.4094235939936062, 0.3949062897112155, 0.40006224970982335, 0.40883441367323736, 0.3895737374171838, 0.40586681182794215, 0.40308247421119997, 0.39537240678802477, 0.41313274228489877, 0.38975387510219983, 0.41515186362446393, 0.4033559409075375, 0.38617451727571417, 0.40566409797542086, 0.40211481420190104, 0.3954705402825181, 0.3923799764377212, 0.4006384799553417, 0.4072302546043632, 0.38486072512593067, 0.41446698122796277, 0.40186875598417104, 0.40448093963216825, 0.4095820109368654, 0.38060890

In [10]:
#========================================================
params = {}# Setting up now for nbra namd calculations

params["T"]          = 300.0
params["sh_method"]  = 1 
params["Boltz_opt"]  = 1 
params["nsteps"]     = nuclear_traj_len
params["init_times"] = [0]
params["ntraj"] = 1

In [13]:
#========================================================
# Setting up now for BLLZ calculations
# Looking on the "SE" populations - Markov chain approach
params["target_space"]       = 1
params["gap_min_exception"]  = 0
params["Boltz_opt_BL"]       = 1     # Option to incorporate hte frustrated hops into BL probabilities
params["evolve_Markov"]      = True  # Rely on the Markov approach
params["evolve_TSH"]         = False # don't care about TSH
params["extend_md"]          = False
params["extend_md_time"]     = 0

In [36]:
def myfunc( istate_index ):

    # At this point, we have the nuclear trajectories for all bases, here, we show just for the mb
    # This is a specific istate, so, here each nuclear trajectory, we need to run the dynamics at this istate
    # For each nuclear trajectory, run dynamics for this istate

    md_time = [ i for i in range( nuclear_traj_len ) ]
    md_time = np.array( md_time ) * units.au2fs

    for nuc_traj_index in range( num_nuclear_trajs ):

        # Get nuc_traj_index hvbis for each basis
        #mb_hvib     = hvib_mb_trajs[ nuc_traj_index ]    
        #istate_mb   = istates_mb_all_trajs[ nuc_traj_index ][ istate_index  ]
        #energies_mb = energies_mb_all_trajs[ nuc_traj_index ]

        mixed_sd_hvib     = hvib_mixed_sd_trajs[ nuc_traj_index ]
        istate_mixed_sd   = istates_mixed_sd_all_trajs[  nuc_traj_index ][ istate_index  ]
        energies_mixed_sd = energies_mixed_sd_all_trajs[ nuc_traj_index ]

        #"""
        ### FSSH
        params["decoherence_method"] = 0
        #outfile = "_mb_fssh_"+str( nuc_traj_index )+"_"+str( istate_index )+".txt"
        #mb_nbra_fssh = run_nbra_namd_wrapper( mb_hvib, istate_mb, outfile, params )
        outfile = "namd/mixed_sd_fssh_"+str( nuc_traj_index )+"_"+str( istate_index )+".txt"
        mixed_sd_nbra_fssh = run_nbra_namd_wrapper( mixed_sd_hvib, istate_mixed_sd, outfile, params )
        #"""
        plt.figure(figsize=(3.2,2.4), dpi=300)
        for state in range(nstates_sp):
            plt.plot(md_time,energies_mixed_sd[state] * units.au2ev,color="gray",linewidth=0.5)
            plt.plot(md_time,mixed_sd_nbra_fssh,color="red",linewidth=2,label="FSSH")

        plt.xlabel("Time (fs)")
        plt.ylabel("Energy (eV)")
        plt.legend()
        plt.tight_layout()

        plt.savefig(f"traj_{nuc_traj_index}_{istate_index}.png")
        """
        ### ID-A
        params["decoherence_method"] = 1
        outfile = "_mb_ida_"+str( nuc_traj_index )+"_"+str( istate_index )+".txt"
        mb_nbra_ida = run_nbra_namd_wrapper( mb_hvib, istate_mb, outfile, params )
        outfile = "_mixed_sd_ida_"+str( nuc_traj_index )+"_"+str( istate_index )+".txt"
        mixed_sd_nbra_ida = run_nbra_namd_wrapper( mixed_sd_hvib, istate_mixed_sd, outfile, params )
        """

        """
        ### mSDM
        params["decoherence_method"] = 2
        outfile = "_mb_msdm_"+str( nuc_traj_index )+"_"+str( istate_index )+".txt"
        mb_nbra_msdm = run_nbra_namd_wrapper( mb_hvib, istate_mb, outfile, params )
        outfile = "_mixed_sd_msdm_"+str( nuc_traj_index )+"_"+str( istate_index )+".txt"
        mixed_sd_nbra_msdm = run_nbra_namd_wrapper( mixed_sd_hvib, istate_mixed_sd, outfile, params )
        """

        """
        ### bllz
        outfile = "_mb_bllz_"+str( nuc_traj_index )+"_"+str( istate_index )+".txt"
        mb_nbra_bllz = run_bllz_wrapper( mb_hvib, istate_mb, outfile, params )
        outfile = "_mixed_sd_bllz_"+str( nuc_traj_index )+"_"+str( istate_index )+".txt"
        mixed_sd_nbra_bllz = run_bllz_wrapper( mixed_sd_hvib, istate_mixed_sd, outfile, params )
        """

        """
        plt.figure(num=None, figsize=(3.21, 2.41), dpi=300, edgecolor='black', frameon=True)
        plt.subplot(1,1,1)
        plt.title('2x2x2 tetragonal, MB '+str( nuc_traj_index )+"_"+str( istate_index )+'', fontsize=10)
        plt.xlabel('Time, fs',   fontsize=10)
        plt.ylabel('Energy, eV', fontsize=10)
        plt.ylim(0,3.2)
        plt.yticks([0,1,2,3])
        for mb_index in range( nstates_mb ):
            plt.plot(md_time, energies_mb[mb_index]*units.au2ev, label="", linewidth=1, color = "gray")
        plt.plot(md_time, mb_nbra_fssh, linewidth=2.5, color="maroon",  label="FSSH")
        plt.tight_layout()
        plt.legend(fontsize=8, ncol=2, loc="upper right")
        plt.savefig("CsPbI3_222tetra_mb_"+str( nuc_traj_index )+"_"+str( istate_index )+".png", dpi=300)
        """

pool = mp.Pool(72)
pool.map( myfunc, list( range( num_istates_to_generate ) ) )
pool.close()
pool.join()

decoherence_method = decoherence_method =  decoherence_method =  00
decoherence_method =  decoherence_method = 
decoherence_method = 0decoherence_method = decoherence_method = 
  decoherence_method = 0   000
decoherence_method = 0
 
decoherence_method = 

0decoherence_method =   decoherence_method = decoherence_method = 
00 decoherence_method = 
 
 0decoherence_method = 00
 decoherence_method = 

decoherence_method = 0 decoherence_method = 
 0 decoherence_method = 
 0decoherence_method = 0decoherence_method = decoherence_method = 
0 
decoherence_method = 
0   decoherence_method = 
00decoherence_method = 0 decoherence_method = 

 
0 decoherence_method = 0
0 decoherence_method = 

0 decoherence_method = 0
 decoherence_method = 
0 decoherence_method = 
 0decoherence_method =  0decoherence_method = 

0 decoherence_method =  
0decoherence_method = 0 
decoherence_method = 
0 decoherence_method = 
0decoherence_method =  
 0decoherence_method =  decoherence_method = 0

0decoherence_method =   




0Decoherence times matrix (fs):
decoherence_method = Decoherence rates matrix (a.u.^-1):
 decoherence_method = 
Decoherence times matrix (fs):
Decoherence times matrix (fs):
Decoherence times matrix (fs):
Decoherence times matrix (a.u. of time):Decoherence times matrix (a.u. of time):
Decoherence times matrix (a.u. of time):0Decoherence rates matrix (a.u.^-1):decoherence_method = 
 0.0000000   0.0030236415  0.0029688981  0.0029112943  0.0029202710  0.0028298054  0.0027608180  0.0026987475  0.0026410426  0.0025845363  0.0025421200  
0.0030236415  0.0000000   0.0011465733  0.0010980674  0.0012729390  0.0013257094  0.0013730719  0.0014725084  0.0016395067  0.0016992221  0.0017084801  
0.0029688981  0.0011465733  0.0000000   0.00065532144  0.00091669504  0.00083953382  0.00088461800  0.0010089507  0.0012293980  0.0012757355  0.0012874000  
0.0029112943  0.0010980674  0.00065532144  0.0000000   0.00063658059  0.00069978122  0.00073112500  0.00089100372  0.0011351555  0.0011575416  0.0011




Decoherence rates matrix (a.u.^-1): 

2.4190000e+08  8.0002870   8.1478040   8.3090192   8.2834777   8.5482910   8.7618961   8.9634173   9.1592616   9.3595126   9.5156800   
8.0002870   2.4190000e+08  21.097648   22.029613   19.003268   18.246835   17.617432   16.427750   14.754438   14.235926   14.158784   
8.1478040   21.097648   2.4190000e+08  36.913183   26.388274   28.813610   27.345137   23.975403   19.676297   18.961611   18.789810   
8.3090192   22.029613   36.913183   2.4190000e+08  37.999902   34.567947   33.085998   27.149157   21.309856   20.897737   20.417737   
8.2834777   19.003268   26.388274   37.999902   2.4190000e+08  47.521552   40.110057   33.248861   25.934053   25.383626   24.589309   
8.5482910   18.246835   28.813610   34.567947   47.521552   2.4190000e+08  54.214781   41.002882   28.215025   27.658975   26.581184   
8.7618961   17.617432   27.345137   33.085998   40.110057   54.214781   2.4190000e+08  53.431961   36.506260   35.116014   32.875641   
8.96341



decoherence_method = Decoherence times matrix (fs):0
decoherence_method = decoherence_method = Decoherence rates matrix (a.u.^-1):Decoherence rates matrix (a.u.^-1):Decoherence times matrix (a.u. of time):Decoherence times matrix (a.u. of time):Decoherence rates matrix (a.u.^-1):
0Decoherence times matrix (a.u. of time):decoherence_method = decoherence_method = 
 decoherence_method = decoherence_method = decoherence_method = 

 Decoherence times matrix (fs):Decoherence times matrix (fs):
 

 decoherence_method = 
Decoherence rates matrix (a.u.^-1):0decoherence_method = 
  0Decoherence times matrix (a.u. of time): 
0Decoherence times matrix (fs):decoherence_method =  Decoherence times matrix (a.u. of time):0.0000000   0.0030236415  0.0029688981  0.0029112943  0.0029202710  0.0028298054  0.0027608180  0.0026987475  0.0026410426  0.0025845363  0.0025421200  
0.0030236415  0.0000000   0.0011465733  0.0010980674  0.0012729390  0.0013257094  0.0013730719  0.0014725084  0.0016395067  0.0016


Decoherence times matrix (fs):0 

Decoherence times matrix (a.u. of time): Decoherence times matrix (fs):
Decoherence rates matrix (a.u.^-1):00

0decoherence_method = 0  Decoherence rates matrix (a.u.^-1):
decoherence_method = 
0
decoherence_method = decoherence_method = 0



Decoherence times matrix (a.u. of time):Decoherence rates matrix (a.u.^-1):Decoherence times matrix (a.u. of time):



0Decoherence rates matrix (a.u.^-1):0Decoherence times matrix (fs):

   
Decoherence rates matrix (a.u.^-1):2.4190000e+08  8.0002870   8.1478040   8.3090192   8.2834777   8.5482910   8.7618961   8.9634173   9.1592616   9.3595126   9.5156800   
8.0002870   2.4190000e+08  21.097648   22.029613   19.003268   18.246835   17.617432   16.427750   14.754438   14.235926   14.158784   
8.1478040   21.097648   2.4190000e+08  36.913183   26.388274   28.813610   27.345137   23.975403   19.676297   18.961611   18.789810   
8.3090192   22.029613   36.913183   2.4190000e+08  37.999902   34.567947   33.085998   

...



Decoherence times matrix (fs):

Decoherence times matrix (a.u. of time):Decoherence times matrix (a.u. of time):

Decoherence rates matrix (a.u.^-1):Decoherence times matrix (a.u. of time):Decoherence times matrix (a.u. of time):Decoherence times matrix (fs):Decoherence times matrix (fs):Decoherence rates matrix (a.u.^-1):Decoherence times matrix (a.u. of time):Decoherence times matrix (a.u. of time):







Decoherence rates matrix (a.u.^-1):Decoherence times matrix (fs):Decoherence times matrix (fs):Decoherence times matrix (a.u. of time):Decoherence rates matrix (a.u.^-1):Decoherence times matrix (a.u. of time):
Decoherence times matrix (fs):Decoherence times matrix (fs):Decoherence times matrix (a.u. of time):Decoherence times matrix (a.u. of time):Decoherence times matrix (a.u. of time):Decoherence times matrix (a.u. of time):Decoherence times matrix (a.u. of time):

Decoherence times matrix (fs):Decoherence times matrix (fs):






Decoherence times matrix (a.u. of time):

Deco

Decoherence times matrix (a.u. of time):Decoherence times matrix (fs):
Decoherence times matrix (a.u. of time):
Decoherence rates matrix (a.u.^-1):Decoherence rates matrix (a.u.^-1):Decoherence times matrix (fs):Decoherence times matrix (a.u. of time):









Decoherence times matrix (fs):Decoherence times matrix (fs):




Decoherence rates matrix (a.u.^-1):Decoherence rates matrix (a.u.^-1):Decoherence rates matrix (a.u.^-1):Decoherence rates matrix (a.u.^-1):Decoherence times matrix (fs):Decoherence rates matrix (a.u.^-1):Decoherence times matrix (a.u. of time):Decoherence times matrix (fs):
Decoherence times matrix (fs):Decoherence times matrix (fs):Decoherence times matrix (a.u. of time):
Decoherence rates matrix (a.u.^-1):




Decoherence rates matrix (a.u.^-1):


Decoherence times matrix (a.u. of time):
Decoherence times matrix (a.u. of time):Decoherence rates matrix (a.u.^-1):Decoherence times matrix (fs):

Decoherence rates matrix (a.u.^-1):

Decoherence rates matrix (a.u.^-1


Decoherence times matrix (a.u. of time):Decoherence times matrix (a.u. of time):Decoherence times matrix (fs):
Decoherence times matrix (a.u. of time):
Decoherence rates matrix (a.u.^-1):Decoherence rates matrix (a.u.^-1):
Decoherence rates matrix (a.u.^-1):
Decoherence times matrix (a.u. of time):


Decoherence rates matrix (a.u.^-1):Decoherence times matrix (fs):

Decoherence times matrix (a.u. of time):Decoherence times matrix (fs):


Decoherence times matrix (fs):
Decoherence times matrix (a.u. of time):Decoherence rates matrix (a.u.^-1):
Decoherence times matrix (fs):
Decoherence times matrix (fs):Decoherence rates matrix (a.u.^-1):

Decoherence rates matrix (a.u.^-1):
Decoherence times matrix (a.u. of time):Decoherence rates matrix (a.u.^-1):Decoherence times matrix (a.u. of time):
Decoherence times matrix (a.u. of time):Decoherence times matrix (fs):

Decoherence times matrix (a.u. of time):Decoherence rates matrix (a.u.^-1):2.4190000e+08  8.0002870   8.1478040   8.3090192   8.



Decoherence times matrix (a.u. of time):




Decoherence times matrix (a.u. of time):
Decoherence times matrix (fs):Decoherence times matrix (fs):Decoherence rates matrix (a.u.^-1):Decoherence times matrix (fs):Decoherence times matrix (fs):

Decoherence times matrix (fs):


Decoherence times matrix (a.u. of time):Decoherence rates matrix (a.u.^-1):Decoherence rates matrix (a.u.^-1):

Decoherence rates matrix (a.u.^-1):Decoherence times matrix (fs):

Decoherence rates matrix (a.u.^-1):Decoherence rates matrix (a.u.^-1):
Decoherence times matrix (a.u. of time):



Decoherence times matrix (fs):Decoherence times matrix (a.u. of time):Decoherence rates matrix (a.u.^-1):



Decoherence times matrix (fs):Decoherence times matrix (fs):Decoherence times matrix (a.u. of time):Decoherence rates matrix (a.u.^-1):Decoherence times matrix (a.u. of time):

Decoherence times matrix (a.u. of time):Decoherence rates matrix (a.u.^-1):Decoherence rates matrix (a.u.^-1):




Decoherence times matrix (f

MaybeEncodingError: Error sending result: '<multiprocessing.pool.ExceptionWithTraceback object at 0xebb5e626b60>'. Reason: 'PicklingError("Can't pickle <class 'Boost.Python.ArgumentError'>: import of module 'Boost.Python' failed")'